# Preprocess

In [14]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import gpytorch
import xarray as xr
import matplotlib.pyplot as plt

## BedMachine data

In [15]:
# Do this only to replicate download and slicing
"""
# Load dataset
bed = xr.open_dataset("BedMachineAntarctica-v3.nc")

# y, x order, slice based on index, reduce to 28 MB
# keep all variables
bed_slice = bed.isel(y = slice(6666, 8666), x = slice(6000, 7400))

slice_filename = './BedMachineAntarctica-v3-slice.nc'
bed_slice.to_netcdf(path = slice_filename)
bed_slice.close()
print ('finished saving')
"""

'\n# Load dataset\nbed = xr.open_dataset("BedMachineAntarctica-v3.nc")\n\n# y, x order, slice based on index, reduce to 28 MB\n# keep all variables\nbed_slice = bed.isel(y = slice(6666, 8666), x = slice(6000, 7400))\n\nslice_filename = \'./BedMachineAntarctica-v3-slice.nc\'\nbed_slice.to_netcdf(path = slice_filename)\nbed_slice.close()\nprint (\'finished saving\')\n'

In [16]:
bed = xr.open_dataset("BedMachineAntarctica-v3-slice.nc")

In [17]:
np.max(bed.x)

<xarray.DataArray 'x' ()>
array(366500, dtype=int32)

In [18]:
### EXPLORE ###
# 3 M values
bed.bed.values
np.max(bed.bed.values)

# Corners
bed.bed.sel(y = 0) # top left value is 68
bed.bed.sel(y = -999500) # bottom left value is -682
bed.bed.sel(x = -333000) # top left again
bed.bed.sel(x = 366500) # top right is -228

# South Pole
bed.bed.sel(y = 0, x = 0) # -27 at origin

<xarray.DataArray 'bed' ()>
array(-27.430664, dtype=float32)
Coordinates:
    x        int32 0
    y        int32 0
Attributes:
    long_name:      bed topography
    standard_name:  bedrock_altitude
    units:          meters
    grid_mapping:   mapping
    source:         IBCSO v2 and Mathieu Morlighem

In [19]:
scene = bed.isel(y = slice(0, 45), x = slice(0, 45))
# 45 * 500m = 22.500 km 
# 50 * 450m  

In [20]:
# to_pandas() contains matrix format
# to_dataframe() keeps multi-index
# .values outputs array which can be converted to tensor

scene_bed_tensor = torch.tensor(scene.bed.values)
scene_sur_tensor = torch.tensor(scene.surface.values)

torch.save(scene_bed_tensor, './torch_data/scene_bed_tensor.pt')
torch.save(scene_sur_tensor, './torch_data/scene_sur_tensor.pt')

In [34]:
scene46 = bed.isel(y = slice(0, 46), x = slice(0, 46))
scene46_sur_tensor = torch.tensor(scene46.surface.values)
torch.save(scene46_sur_tensor, './torch_data/scene46_sur_tensor.pt')

In [29]:
# Increase to 46 to span full area
dims = 46
scene_sur_midpoints46_tensor = torch.cat((torch.tensor(scene.coords["y"].values).unsqueeze(-1).repeat(1, dims).unsqueeze(0),
                              torch.tensor(scene.coords["x"].values).repeat(dims, 1).unsqueeze(0)),
                              dim = 0)

torch.save(scene_sur_midpoints46_tensor, './torch_data/scene_sur_midpoints46_tensor.pt')

In [20]:
fig = px.imshow(scence.bed.values, 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Bed elevation of scene one")
fig.show()

In [22]:
fig = px.imshow(scence.surface.values, 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Surface of scene one")
fig.show()

In [ ]:
# Magnify data because it is too large to visualise
magnification_factor = 10
magnify = torch.nn.AvgPool2d(kernel_size = magnification_factor)
# stride is my default the kernel size

input = torch.tensor(bed.bed.values).unsqueeze(0)
input = input.type(torch.DoubleTensor)
bed_magnified = magnify(input)
bed_magnified.shape

In [ ]:
fig = px.imshow(bed_magnified.squeeze(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Bed elevation of the Transantarctic mountains")
fig.show()

In [ ]:
fig = px.imshow(bed_magnified.squeeze(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Bed elevation of the Transantarctic mountains")
fig.show()

In [ ]:
bed

In [ ]:
fig = px.imshow((bed.mask.values), 
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", 
                title = "Landtypes: 1: ice free land, 2 grounded ice, 3 floating ice")
fig.show()

In [ ]:
bed.source.values

In [ ]:
fig = px.imshow((bed.firn.values), 
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", 
                title = "Firn air content (in meters) correction value")
fig.show()

In [ ]:
bed

In [ ]:
fig = px.imshow((bed.errbed.values), 
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", 
                title = "Firn air content (in meters) correction value")
fig.show()

In [ ]:
# Check: this should hold true
(bed.surface.values - bed.bed.values - bed.thickness.values)

fig = px.imshow((bed.surface.values - bed.bed.values - bed.thickness.values), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Floating ice")
fig.show()

## Ice velocity

In [ ]:
# Do this only to replicate download and slicing

"""
# Load dataset
velo = xr.open_dataset("Antarctica_ice_velocity_2016_2017_1km_v01.nc")

# Get indices 
# list = velo.coords["x"].values == 367000
# [i for i, x in enumerate(list) if x]

# Y: from index 2800 to 3800 (500 m fewer..) [-333000., 366000.]
# X: from index 2467 to 3167 (500 m fewer..) [0., -999000.]

# Reduced to 30 MB
velo_slice = velo.isel(y = slice(2800, 3800), x = slice(2467, 3167))

velo_slice_filename = './Antarctica_ice_velocity_2016_2017_1km_v01_slice.nc'
velo_slice.to_netcdf(path = velo_slice_filename)
velo_slice.close()
print ('finished saving')
"""

In [ ]:
# Load dataset
velo = xr.open_dataset("Antarctica_ice_velocity_2008_2009_1km_v01.nc")

# Get indices 
# list = velo.coords["x"].values == 367000
# [i for i, x in enumerate(list) if x]

# Y: from index 2800 to 3800 (500 m fewer..) [-333000., 366000.]
# X: from index 2467 to 3167 (500 m fewer..) [0., -999000.]

# Reduced to 30 MB
velo_slice = velo.isel(y = slice(2800, 3800), x = slice(2467, 3167))

velo_slice_filename = './Antarctica_ice_velocity_2008_2008_1km_v01_slice.nc'
velo_slice.to_netcdf(path = velo_slice_filename)
velo_slice.close()
print ('finished saving')

In [ ]:
velo = xr.open_dataset("Antarctica_ice_velocity_2008_2009_1km_v01_slice.nc")

In [ ]:
velo.VX

In [ ]:
fig = px.imshow((velo.VY.values), 
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", 
                title = "Ice velocity in x direction (meter/year)")
fig.show()

In [ ]:
velo.CNT.values

In [ ]:
fig = px.imshow((velo.VY.values), 
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", 
                title = "Ice velocity in x direction (meter/year)")
fig.show()

In [ ]:
450*10
# Factor 10 upscale